# Enterprise Finance RAG Assistant using Databricks

## Project Overview

This project demonstrates an enterprise-style **Retrieval Augmented Generation (RAG)** solution built using Databricks.

The solution creates a finance knowledge assistant that can answer natural-language questions using information retrieved from financial and accounting documents.

Instead of relying only on the LLM's pretrained knowledge, the system retrieves relevant information from a curated finance knowledge base and provides it as context to the LLM before generating the response.

### Business Problem

Large organizations maintain many finance-related documents, including:

* Financial reporting procedures
* Revenue recognition guidelines
* Accounting policies
* Budgeting and forecasting guidelines
* Cost allocation policies
* Finance KPI definitions

Finding the correct information manually across these documents can be time-consuming.

The objective of this project is to build a semantic-search-based finance assistant that can retrieve relevant information and generate grounded answers to finance-related questions.

### Example Questions

* What is the definition of operating revenue?
* How is monthly revenue variance calculated?
* What is the process for financial reporting?
* Which policy defines the treatment of a particular expense?
* What are the key financial KPIs used for reporting?

### Technology Stack

* Azure Databricks
* PySpark
* Delta Lake
* Databricks AI Search
* Embeddings
* Vector Search
* Large Language Model (LLM)
* RAG
* Python / SQL

### Project Objective

Build an end-to-end pipeline:

**Documents → Text Extraction → Chunking → Embeddings → Delta Knowledge Base → AI Search → Retrieval → LLM → Finance Answer**


# Solution Architecture

The solution follows an end-to-end RAG architecture.

```text
                 Finance Documents
                        |
                        v
               Document Ingestion
                        |
                        v
                 Text Extraction
                        |
                        v
                     Chunking
                        |
                        v
                Embedding Model
                        |
                        v
             Finance Knowledge Base
                  Delta Table
                        |
                        v
                Databricks AI Search
                        |
                        |
              +---------+---------+
              |                   |
          User Query        Query Embedding
              |                   |
              +---------+---------+
                        |
                        v
                 Semantic Search
                        |
                        v
                 Top-K Chunks
                        |
                        v
                RAG Prompt
             Question + Context
                        |
                        v
                       LLM
                        |
                        v
              Finance Assistant
                  Response
```

### Main Components

**1. Document Ingestion**

Finance documents are collected and loaded into the Databricks environment.

**2. Text Extraction**

Text is extracted from the source documents so that it can be processed by the downstream pipeline.

**3. Chunking**

Large documents are divided into smaller meaningful sections. Each chunk becomes an independent unit for semantic retrieval.

**4. Embedding Generation**

Each text chunk is converted into a numerical vector representation using an embedding model.

**5. Delta Knowledge Base**

The chunks, embeddings and document metadata are stored in a Delta-based knowledge layer.

Example metadata:

* document_id
* document_name
* document_type
* section
* chunk_id
* chunk_text
* embedding

**6. Databricks AI Search**

AI Search provides an optimized search layer over the knowledge base and retrieves relevant chunks based on the user's query.

**7. RAG Generation**

The retrieved chunks are provided to the LLM as additional context along with the user's question.

**8. Final Response**

The LLM uses the retrieved context to generate a natural-language answer.


###Check files in volumes

In [0]:
display(dbutils.fs.ls('/Volumes/workspace/finance_genai/finance_documents'))

path,name,size,modificationTime
dbfs:/Volumes/workspace/finance_genai/finance_documents/Budget & Forecasting Guidelines.pdf,Budget & Forecasting Guidelines.pdf,108244,1789375425000
dbfs:/Volumes/workspace/finance_genai/finance_documents/Finance Reporting Policy.pdf,Finance Reporting Policy.pdf,119985,1789375425000
dbfs:/Volumes/workspace/finance_genai/finance_documents/Monthly Revenue Reporting Procedure.pdf,Monthly Revenue Reporting Procedure.pdf,111897,1789375425000
dbfs:/Volumes/workspace/finance_genai/finance_documents/Revenue Recognition Policy.pdf,Revenue Recognition Policy.pdf,104979,1789375425000


###PDF text extraction.

In [0]:
%pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 150.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

###here each pdf is read and pagewise text is extracted and stored

In [0]:
import pymupdf
import os

volume_path = "/Volumes/workspace/finance_genai/finance_documents"

pdf_files = [
    "Finance Reporting Policy.pdf",
    "Revenue Recognition Policy.pdf",
    "Monthly Revenue Reporting Procedure.pdf",
    "Budget & Forecasting Guidelines.pdf"
]

documents = []

for file_name in pdf_files:
    file_path = os.path.join(volume_path, file_name)

    pdf = pymupdf.open(file_path)

    for page_number, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()

        if text:
            documents.append({
                "document_name": file_name,
                "page_number": page_number,
                "text": text
            })

    pdf.close()

print(f"Total pages extracted: {len(documents)}")

Total pages extracted: 28


In [0]:
display(spark.createDataFrame(documents))

document_name,page_number,text
Finance Reporting Policy.pdf,1,"Finance Reporting Policy Company: Northstar Energy Services Ltd.​ Document ID: FIN-POL-001​ Version: 3.0​ Effective Date: 1 January 2026​ Document Owner: Finance Controlling Project Note: This is a fictional enterprise document created for the Enterprise Finance RAG Assistant project. 1. Purpose This policy establishes the principles, responsibilities, reporting calendar, and control requirements for monthly and quarterly financial reporting at Northstar Energy Services Ltd. The objective is to ensure that financial information used by management is: ●​ Accurate ●​ Complete ●​ Consistent ●​ Timely ●​ Traceable to approved source systems ●​ Supported by appropriate reconciliations and documentation This policy provides the overall framework for the Monthly Revenue Reporting Procedure and the Budget & Forecasting Guidelines. 2. Scope This policy applies to: ●​ Finance Controlling ●​ Business Finance teams ●​ Finance Data Operations ●​ Business Unit Finance Owners ●​ Finance Controllers ●​ Employees responsible for financial reporting inputs"
Finance Reporting Policy.pdf,2,"The policy covers management reporting for revenue, operating costs, budget, forecast, and selected financial KPIs. 3. Reporting Periods Northstar follows a monthly management reporting cycle. The standard reporting periods are: Reporting Cycle Period Target Completion Monthly Calendar month Business Day 5 Quarterly Three-month quarter Business Day 8 Annual Full fiscal year According to annual close calendar The monthly close process begins after the source-system reporting cut-off. The reporting period must be clearly identified in every reporting dataset and management report. 4. Monthly Close Process The monthly financial close consists of the following major activities: 1.​ Source-system data extraction 2.​ Data quality validation 3.​ Revenue validation 4.​ Cost validation 5.​ Accrual review 6.​ Intercompany reconciliation 7.​ Revenue reconciliation 8.​ Financial adjustments 9.​ Actual-versus-budget analysis 10.​Actual-versus-forecast analysis 11.​Management commentary 12.​Finance review"
Finance Reporting Policy.pdf,3,"13.​Final management approval Detailed operational steps for revenue reporting are defined in the Monthly Revenue Reporting Procedure. 5. Revenue Definition For management reporting purposes, revenue represents income recognized from customer contracts in accordance with the Revenue Recognition Policy. Operating revenue is revenue generated from the company's ordinary customer activities. Northstar classifies operating revenue into: ●​ Energy Sales ●​ Service Revenue ●​ Other Operating Revenue The definitions and recognition rules for these categories are maintained in the Revenue Recognition Policy. 6. Financial Reporting Pack The monthly management reporting pack must contain, where applicable: ●​ Actual revenue ●​ Approved budget revenue ●​ Latest forecast revenue ●​ Revenue variance ●​ Revenue variance percentage ●​ Operating costs ●​ Operating cost variance ●​ Operating margin ●​ Selected operational KPIs ●​ Management commentary for material variances The reporting pack must use approved actuals from the monthly reporting process."
Finance Reporting Policy.pdf,4,"7. Revenue Variance Revenue variance is calculated as: Revenue Variance = Actual Revenue − Budget Revenue Revenue variance percentage is calculated as: Revenue Variance % = (Actual Revenue − Budget Revenue) / Budget Revenue × 100 A positive variance indicates actual revenue is above budget. A negative variance indicates actual revenue is below budget. Example Assume: ●​ Actual Revenue = USD 18 million ●​ Budget Revenue = USD 20 million Revenue variance: USD 18M − USD 20M = USD -2M Revenue variance percentage: (-2M / 20M) × 100 = -10% Actual revenue is therefore 10% below budget. 8. Material Variance Threshold A revenue variance is considered material for management commentary when: ●​ The absolute percentage varia

##store it in delta table

In [0]:
from pyspark.sql.functions import col, current_timestamp, monotonically_increasing_id
documents_df=spark.createDataFrame(documents)


documents_df = (
    documents_df
    .withColumn("document_id", monotonically_increasing_id())
    .withColumn("ingestion_timestamp", current_timestamp())
)

documents_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.finance_genai.finance_documents_raw")

In [0]:
documents_df=spark.read.table("workspace.finance_genai.finance_documents_raw")

### here each page text is read and broken into chunks of 400 words and stored

0- st to 400- end
350 - st to (350+400) 750 - end
350+50 = 400 - st to (400+400) - 800 - end


In [0]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import ArrayType, StringType
import re

def create_chunks(text, chunk_size=400, overlap=50):
    words = re.findall(r'\S+', text)

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

chunk_udf = udf(
    create_chunks,
    ArrayType(StringType())
)

chunks_df = (
    documents_df
    .withColumn("chunks", chunk_udf(col("text")))
    .select(
        "document_name",
        "page_number",
        "chunks"
    )
)

display(chunks_df)

document_name,page_number,chunks
Finance Reporting Policy.pdf,1,"List(Finance Reporting Policy Company: Northstar Energy Services Ltd.​ Document ID: FIN-POL-001​ Version: 3.0​ Effective Date: 1 January 2026​ Document Owner: Finance Controlling Project Note: This is a fictional enterprise document created for the Enterprise Finance RAG Assistant project. 1. Purpose This policy establishes the principles, responsibilities, reporting calendar, and control requirements for monthly and quarterly financial reporting at Northstar Energy Services Ltd. The objective is to ensure that financial information used by management is: ●​ Accurate ●​ Complete ●​ Consistent ●​ Timely ●​ Traceable to approved source systems ●​ Supported by appropriate reconciliations and documentation This policy provides the overall framework for the Monthly Revenue Reporting Procedure and the Budget & Forecasting Guidelines. 2. Scope This policy applies to: ●​ Finance Controlling ●​ Business Finance teams ●​ Finance Data Operations ●​ Business Unit Finance Owners ●​ Finance Controllers ●​ Employees responsible for financial reporting inputs)"
Finance Reporting Policy.pdf,2,"List(The policy covers management reporting for revenue, operating costs, budget, forecast, and selected financial KPIs. 3. Reporting Periods Northstar follows a monthly management reporting cycle. The standard reporting periods are: Reporting Cycle Period Target Completion Monthly Calendar month Business Day 5 Quarterly Three-month quarter Business Day 8 Annual Full fiscal year According to annual close calendar The monthly close process begins after the source-system reporting cut-off. The reporting period must be clearly identified in every reporting dataset and management report. 4. Monthly Close Process The monthly financial close consists of the following major activities: 1.​ Source-system data extraction 2.​ Data quality validation 3.​ Revenue validation 4.​ Cost validation 5.​ Accrual review 6.​ Intercompany reconciliation 7.​ Revenue reconciliation 8.​ Financial adjustments 9.​ Actual-versus-budget analysis 10.​Actual-versus-forecast analysis 11.​Management commentary 12.​Finance review)"
Finance Reporting Policy.pdf,3,"List(13.​Final management approval Detailed operational steps for revenue reporting are defined in the Monthly Revenue Reporting Procedure. 5. Revenue Definition For management reporting purposes, revenue represents income recognized from customer contracts in accordance with the Revenue Recognition Policy. Operating revenue is revenue generated from the company's ordinary customer activities. Northstar classifies operating revenue into: ●​ Energy Sales ●​ Service Revenue ●​ Other Operating Revenue The definitions and recognition rules for these categories are maintained in the Revenue Recognition Policy. 6. Financial Reporting Pack The monthly management reporting pack must contain, where applicable: ●​ Actual revenue ●​ Approved budget revenue ●​ Latest forecast revenue ●​ Revenue variance ●​ Revenue variance percentage ●​ Operating costs ●​ Operating cost variance ●​ Operating margin ●​ Selected operational KPIs ●​ Management commentary for material variances The reporting pack must use approved actuals from the monthly reporting process.)"
Finance Reporting Policy.pdf,4,"List(7. Revenue Variance Revenue variance is calculated as: Revenue Variance = Actual Revenue − Budget Revenue Revenue variance percentage is calculated as: Revenue Variance % = (Actual Revenue − Budget Revenue) / Budget Revenue × 100 A positive variance indicates actual revenue is above budget. A negative variance indicates actual revenue is below budget. Example Assume: ●​ Actual Revenue = USD 18 million ●​ Budget Revenue = USD 20 million Revenue variance: USD 18M − USD 20M = USD -2M Revenue variance percentage: (-2M / 20M) × 100 = -10% Actual revenue is therefore 10% below budget. 8. Material Variance Threshold A revenue variance is considered material for management commentary when: ●​ The 

In [0]:
from pyspark.sql.functions import explode, row_number,monotonically_increasing_id
from pyspark.sql.window import Window

chunks_df = (
    chunks_df
    .withColumn("chunk", explode(col("chunks")))
    .drop("chunks")
)


display(chunks_df)

document_name,page_number,chunk
Finance Reporting Policy.pdf,1,"Finance Reporting Policy Company: Northstar Energy Services Ltd.​ Document ID: FIN-POL-001​ Version: 3.0​ Effective Date: 1 January 2026​ Document Owner: Finance Controlling Project Note: This is a fictional enterprise document created for the Enterprise Finance RAG Assistant project. 1. Purpose This policy establishes the principles, responsibilities, reporting calendar, and control requirements for monthly and quarterly financial reporting at Northstar Energy Services Ltd. The objective is to ensure that financial information used by management is: ●​ Accurate ●​ Complete ●​ Consistent ●​ Timely ●​ Traceable to approved source systems ●​ Supported by appropriate reconciliations and documentation This policy provides the overall framework for the Monthly Revenue Reporting Procedure and the Budget & Forecasting Guidelines. 2. Scope This policy applies to: ●​ Finance Controlling ●​ Business Finance teams ●​ Finance Data Operations ●​ Business Unit Finance Owners ●​ Finance Controllers ●​ Employees responsible for financial reporting inputs"
Finance Reporting Policy.pdf,2,"The policy covers management reporting for revenue, operating costs, budget, forecast, and selected financial KPIs. 3. Reporting Periods Northstar follows a monthly management reporting cycle. The standard reporting periods are: Reporting Cycle Period Target Completion Monthly Calendar month Business Day 5 Quarterly Three-month quarter Business Day 8 Annual Full fiscal year According to annual close calendar The monthly close process begins after the source-system reporting cut-off. The reporting period must be clearly identified in every reporting dataset and management report. 4. Monthly Close Process The monthly financial close consists of the following major activities: 1.​ Source-system data extraction 2.​ Data quality validation 3.​ Revenue validation 4.​ Cost validation 5.​ Accrual review 6.​ Intercompany reconciliation 7.​ Revenue reconciliation 8.​ Financial adjustments 9.​ Actual-versus-budget analysis 10.​Actual-versus-forecast analysis 11.​Management commentary 12.​Finance review"
Finance Reporting Policy.pdf,3,"13.​Final management approval Detailed operational steps for revenue reporting are defined in the Monthly Revenue Reporting Procedure. 5. Revenue Definition For management reporting purposes, revenue represents income recognized from customer contracts in accordance with the Revenue Recognition Policy. Operating revenue is revenue generated from the company's ordinary customer activities. Northstar classifies operating revenue into: ●​ Energy Sales ●​ Service Revenue ●​ Other Operating Revenue The definitions and recognition rules for these categories are maintained in the Revenue Recognition Policy. 6. Financial Reporting Pack The monthly management reporting pack must contain, where applicable: ●​ Actual revenue ●​ Approved budget revenue ●​ Latest forecast revenue ●​ Revenue variance ●​ Revenue variance percentage ●​ Operating costs ●​ Operating cost variance ●​ Operating margin ●​ Selected operational KPIs ●​ Management commentary for material variances The reporting pack must use approved actuals from the monthly reporting process."
Finance Reporting Policy.pdf,4,"7. Revenue Variance Revenue variance is calculated as: Revenue Variance = Actual Revenue − Budget Revenue Revenue variance percentage is calculated as: Revenue Variance % = (Actual Revenue − Budget Revenue) / Budget Revenue × 100 A positive variance indicates actual revenue is above budget. A negative variance indicates actual revenue is below budget. Example Assume: ●​ Actual Revenue = USD 18 million ●​ Budget Revenue = USD 20 million Revenue variance: USD 18M − USD 20M = USD -2M Revenue variance percentage: (-2M / 20M) × 100 = -10% Actual revenue is therefore 10% below budget. 8. Material Variance Threshold A revenue variance is considered material for management commentary when: ●​ The absolute percentage vari

### store it in delta table

In [0]:
chunks_df = (
    chunks_df
    .withColumn("chunk_id", monotonically_increasing_id())
    .select(
        "chunk_id",
        "document_name",
        "page_number",
        "chunk"
    )
)

chunks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.finance_genai.finance_document_chunks")


### chunk table - Change Data Feed enabled, because a standard Delta Sync AI Search index needs it.

In [0]:
spark.sql("""
ALTER TABLE workspace.finance_genai.finance_document_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

DataFrame[]

In [0]:
display(
    spark.sql("""
        SHOW TBLPROPERTIES workspace.finance_genai.finance_document_chunks
    """)
)

key,value
delta.enableChangeDataFeed,true
delta.enableDeletionVectors,true
delta.feature.appendOnly,supported
delta.feature.changeDataFeed,supported
delta.feature.deletionVectors,supported
delta.feature.invariants,supported
delta.minReaderVersion,3
delta.minWriterVersion,7
delta.parquet.compression.codec,zstd
delta.parquet.format.version,2.12.0


### install databricks ai search - after we create ai search index

In [0]:
%pip install databricks-ai-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-0fc3bd26-3c8a-47bd-8097-a44dbd7f05c0
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-0fc3bd26-3c8a-47bd-8097-a44dbd7f05c0
    Can't uninstall 'mlflow-skinny'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Connect Python to your AI Search index

vsc
 ↓
AI Search service

index
 ↓
your specific finance_document_chunks_index

In [0]:
from databricks.ai_search.client import AISearchClient

vsc = AISearchClient()

index = vsc.get_index(
    index_name="workspace.finance_genai.finance_document_chunks_index"
)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


### Retrieve relevant chunks

In [0]:
question = "What are the criteria for recognizing revenue?"

results = index.similarity_search(
    query_text=question,
    columns=["chunk", "document_name"],
    num_results=3,
    query_type="hybrid"
)

print(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{'manifest': {'column_count': 3, 'columns': [{'name': 'chunk'}, {'name': 'document_name'}, {'name': 'score'}]}, 'result': {'row_count': 3, 'data_array': [['Assume an operational system records USD 10M of customer activity during December, but the applicable revenue recognition criteria are satisfied in January. The USD 10M should not automatically be included in December recognized revenue merely because the operational transaction exists. The appropriate recognition period must be determined using the contract and performance-obligation assessment. 15. Relationship With Other Documents The Finance Reporting Policy establishes the overall reporting framework. This document establishes the revenue recognition rules used within that framework. The Monthly Revenue Reporting Procedur

### Extract the rows and Build the context
result
 └── data_array
      ├── [chunk, document_name, score]
      ├── [chunk, document_name, score]
      └── [chunk, document_name, score]


Context = information retrieved from your documents that we give to the LLM.

In [0]:
rows = results["result"]["data_array"]

context = "\n\n".join(
    [
        f"Source: {row[1]}\nContent: {row[0]}"
        for row in rows
    ]
)

print(context)

Source: Revenue Recognition Policy.pdf
Content: Assume an operational system records USD 10M of customer activity during December, but the applicable revenue recognition criteria are satisfied in January. The USD 10M should not automatically be included in December recognized revenue merely because the operational transaction exists. The appropriate recognition period must be determined using the contract and performance-obligation assessment. 15. Relationship With Other Documents The Finance Reporting Policy establishes the overall reporting framework. This document establishes the revenue recognition rules used within that framework. The Monthly Revenue Reporting Procedure operationalizes these rules by defining the extraction, validation, classification, reconciliation, and reporting process. The Budget & Forecasting Guidelines use the resulting actual revenue values for performance comparison and forecasting. 16. Document Control Owner: Financial Accounting​ Review Frequency: Annua

AI Search \
    ↓
"What information should I give the LLM?" 

Llama \
    ↓
"How should I formulate the answer?" 

1.Build the LLM prompt \
2.Tell the LLM its role \
3.Give it the user's question \
4.Give it the retrieved context \
5.The anti-hallucination instruction \
6.Send the prompt to Llama

In [0]:
from pyspark.sql.functions import expr

prompt = f"""
You are a finance policy assistant.

Answer the user's question ONLY using the provided finance policy context.

If the context does not contain enough information to answer the question, say:
"I could not find enough information in the provided finance policies."

Do not invent or assume information.

User question:
{question}

Finance policy context:
{context}

Provide a concise answer and mention the relevant policy document(s) when possible.
"""

result = spark.sql(f"""
SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    {repr(prompt)}
) AS answer
""")

display(result)

answer
"The criteria for recognizing revenue include contract period, promised goods or services, performance obligations, transaction price, variable consideration, pricing adjustments, and timing of performance. These criteria are outlined in the Revenue Recognition Policy.pdf. Additionally, the policy states that contract-specific requirements take precedence over generic assumptions, and the recognition approach must be consistent with the nature of the contract and the approved accounting assessment."


reusable function

In [0]:
def ask_finance_policy(question):

    # 1. Retrieve relevant chunks
    results = index.similarity_search(
        query_text=question,
        columns=["chunk", "document_name"],
        num_results=3,
        query_type="hybrid"
    )

    # 2. Extract retrieved rows
    rows = results["result"]["data_array"]

    # 3. Build context
    context = "\n\n".join(
        [
            f"Source: {row[1]}\nContent: {row[0]}"
            for row in rows
        ]
    )

    # 4. Build grounded prompt
    prompt = f"""
You are a finance policy assistant.

Answer the user's question ONLY using the provided finance policy context.

If the context does not contain enough information to answer the question, say:
"I could not find enough information in the provided finance policies."

Do not invent or assume information.

User question:
{question}

Finance policy context:
{context}

Provide a concise answer and mention the relevant policy document(s).
"""

    # 5. Generate answer
    result = spark.sql(f"""
    SELECT ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        {repr(prompt)}
    ) AS answer
    """)

    return result.collect()[0]["answer"]

In [0]:
display(ask_finance_policy(
    "How is revenue recognized for services performed over time?"
))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


'Revenue is recognized for services performed over time using the approved measure of performance for the reporting period, as stated in the Revenue Recognition Policy (FIN-POL-002, Version 2.2). For example, if a service contract requires monthly service delivery and the monthly performance obligation has been satisfied, the applicable monthly revenue may be recognized subject to contract terms and approved adjustments (Section 7, Revenue Recognized Over Time).'

In [0]:
display(ask_finance_policy(
    "what is the calculation for forecast variance"
))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


'The calculation for forecast variance is: Forecast Variance = Actual Revenue - Latest Forecast. This is stated in the "Monthly Revenue Reporting Procedure.pdf" and also referenced in the "Budget & Forecasting Guidelines.pdf".'

In [0]:
display(ask_finance_policy(
    "what should i do if i find out there are material unresolved differences?"
))

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


'If you find out there are material unresolved differences, you should escalate them to the Finance Controller. (Refer to Finance Reporting Policy.pdf, section 10, and Revenue Recognition Policy.pdf)'